# Feature Engineering for Advanced EDA

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_parquet("../data/processed/taxi_trip_cleaned.parquet")

## Trip Efficiency Features

In [4]:
df["fare_per_minute"] = np.where(
    df["trip_duration_minutes"] > 0,
    df["fare_amount"] / df["trip_duration_minutes"],
    np.nan
)

In [15]:
'''df["total_per_minute"] = np.where(
    df["trip_duration_minutes"] > 0,
    df["total_amount"] / df["trip_duration_minutes"],
    np.nan
)'''
df["total_per_minute"] = (
    df["total_amount"] / df["trip_duration_minutes"]
)

In [6]:
df["total_per_mile"] = np.where(
    df["trip_distance"] > 0,
    df["total_amount"] / df["trip_distance"],
    np.nan
)

## To be protected from anomalies

In [16]:
df_feature_analysis = df[
    (df["fare_amount"] > 0) &
    (df["trip_distance"] > 0) &
    (df["trip_duration_minutes"] > 0) &
    (~df["extreme_distance_flag"]) &
    (~df["long_duration_flag"])
].copy()

In [17]:
df_feature_analysis.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,pickup_weekday,pickup_month,pickup_year,pickup_period,average_speed_mph,fare_per_mile,is_weekend,fare_per_minute,total_per_mile,total_per_minute
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,3,1,2026,Night,10.486486,7.422680,False,1.297297,16.350515,2.857658
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,...,3,1,2026,Night,9.446064,8.777778,False,1.381924,15.166667,2.387755
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,...,3,1,2026,Night,9.455910,7.642857,False,1.204503,13.535714,2.133208
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,...,3,1,2026,Night,7.822430,6.935484,False,0.904206,9.956989,1.298131
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,...,3,1,2026,Night,9.600000,6.250000,False,1.000000,10.694444,1.711111


## Inspect the new features

In [18]:
feature_cols = [
    "fare_per_minute",
    "total_per_minute",
    "fare_per_mile",
    "total_per_mile"
]

print(
    df_feature_analysis[feature_cols].describe()
)

       fare_per_minute  total_per_minute  fare_per_mile  total_per_mile
count     3.105733e+06      3.105733e+06   3.105733e+06    3.105733e+06
mean      2.361483e+00      3.323848e+00   3.058198e+01    4.059034e+01
std       3.242554e+01      3.769424e+01   2.867506e+02    3.349801e+02
min       2.676898e-05      0.000000e+00   1.282051e-03    0.000000e+00
25%       1.001669e+00      1.500000e+00   6.368159e+00    9.403226e+00
50%       1.158177e+00      1.831304e+00   7.986207e+00    1.278261e+01
75%       1.405995e+00      2.324034e+00   1.036145e+01    1.768862e+01
max       6.000000e+03      6.462000e+03   4.900000e+04    4.910000e+04


## create Trip Categories

In [19]:
distance_bins = [
    0,
    2,
    5,
    10,
    np.inf
]

distance_labels = [
    "Short",
    "Medium",
    "Long",
    "Very Long"
]

df["distance_category"] = pd.cut(
    df["trip_distance"],
    bins=distance_bins,
    labels=distance_labels,
    right=False
)

## Check the distribution

In [20]:
distance_distribution = (
    df["distance_category"]
    .value_counts(dropna=False)
    .sort_index()
)

print(distance_distribution)

distance_category
Short        2004188
Medium       1023775
Long          415690
Very Long     281235
Name: count, dtype: int64


In [21]:
distance_percentage = (
    df["distance_category"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print(distance_percentage)

distance_category
Short        53.805322
Medium       27.484719
Long         11.159799
Very Long     7.550160
Name: proportion, dtype: float64


## investigate the extreme ratios

In [23]:
df_feature_analysis[
    [
        "fare_amount",
        "total_amount",
        "trip_distance",
        "trip_duration_minutes",
        "fare_per_minute",
        "total_per_minute"
    ]
].sort_values(
    "fare_per_minute",
    ascending=False
).head(20)

,fare_amount,total_amount,trip_distance,trip_duration_minutes,fare_per_minute,total_per_minute
670319,800.00,801.00,0.03,0.133333,6000.000000,6007.500000
2135862,540.00,541.00,0.06,0.100000,5400.000000,5410.000000
1643368,88.75,107.70,0.02,0.016667,5325.000000,6462.000000
1235902,85.00,103.20,0.21,0.016667,5100.000000,6192.000000
366576,84.00,102.00,0.02,0.016667,5040.000000,6120.000000
233497,330.00,351.00,0.06,0.066667,4950.000000,5265.000000
275600,490.00,491.00,0.01,0.100000,4900.000000,4910.000000
2264612,399.00,400.00,0.01,0.083333,4788.000000,4800.000000
492597,452.00,453.00,0.01,0.100000,4520.000000,4530.000000
963427,421.00,422.00,0.01,0.100000,4210.000000,4220.000000


In [25]:
print(
    df_feature_analysis["trip_duration_minutes"].describe(
        percentiles=[0.001, 0.005, 0.01, 0.05, 0.25, 0.50, 0.75]
    )
)

count    3.105733e+06
mean     1.434471e+01
std      2.068211e+01
min      1.666667e-02
0.1%     1.333333e-01
0.5%     1.383333e+00
1%       2.016667e+00
5%       3.766667e+00
25%      7.766667e+00
50%      1.220000e+01
75%      1.818333e+01
max      1.439550e+03
Name: trip_duration_minutes, dtype: float64


In [26]:
print(
    df_feature_analysis["trip_distance"].describe(
        percentiles=[0.001, 0.005, 0.01, 0.05, 0.25, 0.50, 0.75]
    )
)

count    3.105733e+06
mean     2.193522e+00
std      1.658128e+00
min      1.000000e-02
0.1%     1.000000e-02
0.5%     1.000000e-02
1%       1.000000e-01
5%       4.900000e-01
25%      1.000000e+00
50%      1.680000e+00
75%      2.870000e+00
max      7.820000e+00
Name: trip_distance, dtype: float64


In [27]:
short_duration_counts = {
    "< 0.1 min": (df_feature_analysis["trip_duration_minutes"] < 0.1).sum(),
    "< 0.5 min": (df_feature_analysis["trip_duration_minutes"] < 0.5).sum(),
    "< 1 min": (df_feature_analysis["trip_duration_minutes"] < 1).sum(),
    "< 2 min": (df_feature_analysis["trip_duration_minutes"] < 2).sum()
}

print(short_duration_counts)

{'< 0.1 min': np.int64(1623), '< 0.5 min': np.int64(6985), '< 1 min': np.int64(10566), '< 2 min': np.int64(30233)}


In [28]:
short_distance_counts = {
    "< 0.05 miles": (df_feature_analysis["trip_distance"] < 0.05).sum(),
    "< 0.1 miles": (df_feature_analysis["trip_distance"] < 0.1).sum(),
    "< 0.25 miles": (df_feature_analysis["trip_distance"] < 0.25).sum(),
    "< 0.5 miles": (df_feature_analysis["trip_distance"] < 0.5).sum()
}

print(short_distance_counts)

{'< 0.05 miles': np.int64(26085), '< 0.1 miles': np.int64(30513), '< 0.25 miles': np.int64(47455), '< 0.5 miles': np.int64(159862)}


In [29]:
short_trip_candidates = df_feature_analysis[
    (df_feature_analysis["trip_duration_minutes"] < 1) &
    (df_feature_analysis["trip_distance"] < 0.25) &
    (df_feature_analysis["fare_amount"] > 50)
].copy()

print("Suspicious short-trip candidates:", len(short_trip_candidates))

Suspicious short-trip candidates: 4035


In [30]:
short_trip_candidates[
    [
        "fare_amount",
        "total_amount",
        "trip_distance",
        "trip_duration_minutes",
        "fare_per_minute",
        "fare_per_mile"
    ]
].sort_values(
    "fare_amount",
    ascending=False
).head(20)

,fare_amount,total_amount,trip_distance,trip_duration_minutes,fare_per_minute,fare_per_mile
670319,800.0,801.00,0.03,0.133333,6000.000000,26666.666667
2135862,540.0,541.00,0.06,0.100000,5400.000000,9000.000000
2156424,498.0,499.00,0.02,0.150000,3320.000000,24900.000000
275600,490.0,491.00,0.01,0.100000,4900.000000,49000.000000
492597,452.0,453.00,0.01,0.100000,4520.000000,45200.000000
963427,421.0,422.00,0.01,0.100000,4210.000000,42100.000000
2140203,412.0,413.00,0.04,0.116667,3531.428571,10300.000000
2264613,399.0,400.00,0.01,0.133333,2992.500000,39900.000000
2264612,399.0,400.00,0.01,0.083333,4788.000000,39900.000000
1734446,399.0,400.00,0.09,0.133333,2992.500000,4433.333333


In [31]:
df["suspicious_efficiency_flag"] = (
    (df["trip_duration_minutes"] < 1) &
    (df["trip_distance"] < 0.25) &
    (df["fare_amount"] > 50)
)

In [32]:
print(
    df["suspicious_efficiency_flag"].value_counts()
)

suspicious_efficiency_flag
False    3713109
True       11779
Name: count, dtype: int64


In [33]:
print(
    df["suspicious_efficiency_flag"].mean() * 100
)

0.3162242730519683


In [34]:
df_efficiency_analysis = df[
    (df["fare_amount"] > 0) &
    (df["trip_distance"] > 0) &
    (df["trip_duration_minutes"] > 0) &
    (~df["extreme_distance_flag"]) &
    (~df["long_duration_flag"]) &
    (~df["suspicious_efficiency_flag"])
].copy()

In [35]:
feature_cols = [
    "fare_per_minute",
    "total_per_minute",
    "fare_per_mile",
    "total_per_mile"
]

print(
    df_efficiency_analysis[feature_cols].describe()
)

       fare_per_minute  total_per_minute  fare_per_mile  total_per_mile
count     3.101698e+06      3.101698e+06   3.101698e+06    3.101698e+06
mean      1.420860e+00      2.224264e+00   2.659962e+01    3.598392e+01
std       7.089729e+00      8.611772e+00   2.391675e+02    2.828530e+02
min       2.676898e-05      0.000000e+00   1.282051e-03    0.000000e+00
25%       1.001560e+00      1.499218e+00   6.367521e+00    9.398896e+00
50%       1.157676e+00      1.830508e+00   7.979798e+00    1.277273e+01
75%       1.403848e+00      2.321053e+00   1.034833e+01    1.765432e+01
max       2.653200e+03      3.354000e+03   2.250000e+04    2.277500e+04


In [36]:
distance_bins = [
    0,
    2,
    5,
    10,
    np.inf
]

distance_labels = [
    "Short",
    "Medium",
    "Long",
    "Very Long"
]

df["distance_category"] = pd.cut(
    df["trip_distance"],
    bins=distance_bins,
    labels=distance_labels,
    right=False
)

In [37]:
print(
    df["distance_category"]
    .value_counts()
    .sort_index()
)

distance_category
Short        2004188
Medium       1023775
Long          415690
Very Long     281235
Name: count, dtype: int64


In [38]:
print(
    df["distance_category"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

distance_category
Short        53.805322
Medium       27.484719
Long         11.159799
Very Long     7.550160
Name: proportion, dtype: float64


#### The ratio features are strongly right-skewed. A small number of extremely short, high-fare trips substantially inflate the mean, while the median remains relatively stable. This demonstrates why robust statistics such as the median are important when analyzing efficiency-based ratios.

In [39]:
ratio_cols = [
    "fare_per_minute",
    "total_per_minute",
    "fare_per_mile",
    "total_per_mile"
]

print(
    df_efficiency_analysis[ratio_cols]
    .quantile([0.90, 0.95, 0.99, 0.995, 0.999])
)

       fare_per_minute  total_per_minute  fare_per_mile  total_per_mile
0.900         1.771654          3.046154      14.107143       24.605579
0.950         2.125000          3.721519      18.372093       31.352941
0.990         3.795918          6.176477      77.095238      120.714286
0.995         5.273441          8.330100    1460.000000     1893.000000
0.999        25.714286         38.979671    3898.000000     4551.000000


In [40]:
print(
    df_efficiency_analysis[
        df_efficiency_analysis["fare_per_minute"] > 100
    ][
        [
            "fare_amount",
            "total_amount",
            "trip_distance",
            "trip_duration_minutes",
            "fare_per_minute",
            "fare_per_mile"
        ]
    ]
    .sort_values("fare_per_minute", ascending=False)
    .head(20)
)

         fare_amount  total_amount  trip_distance  trip_duration_minutes  \
1354047        88.44        111.80           0.37               0.033333   
1586309        87.50        106.80           0.25               0.033333   
1055949        87.44        110.55           0.68               0.033333   
1025462        87.00        105.60           0.34               0.033333   
1109334        70.00         97.80           0.40               0.033333   
825526        275.00        345.00           0.50               0.150000   
1683304        88.75         91.75           2.54               0.050000   
1587923        88.75         99.75           0.65               0.050000   
414087         87.44         93.44           1.37               0.050000   
340896         87.00        105.60           0.58               0.050000   
795828         86.00         87.00           0.34               0.050000   
2405993        85.00        103.20           3.47               0.050000   
2578327     

In [41]:
df["very_short_duration_flag"] = (
    df["trip_duration_minutes"] < 1
)

In [42]:
print(df["very_short_duration_flag"].value_counts())
print(df["very_short_duration_flag"].mean() * 100)

very_short_duration_flag
False    3641142
True       83746
Name: count, dtype: int64
2.248282364463039


In [43]:
df.loc[
    df["very_short_duration_flag"],
    [
        "fare_amount",
        "trip_distance",
        "trip_duration_minutes",
        "fare_per_minute",
        "fare_per_mile"
    ]
].describe()

,fare_amount,trip_distance,trip_duration_minutes,fare_per_minute,fare_per_mile
count,83746.000000,83746.000000,83746.000000,38677.000000,56586.000000
mean,23.640298,1.368504,0.126404,284.751579,246.080514
std,39.973787,2.888572,0.213681,555.436430,1274.679965
min,-990.000000,0.000000,0.000000,-11880.000000,-39900.000000
25%,5.800000,0.000000,0.000000,7.200000,6.333333
50%,12.100000,0.500000,0.000000,60.000000,8.169014
75%,25.400000,1.560000,0.150000,420.000000,12.333333
max,2500.000000,107.160000,0.983333,11880.000000,49000.000000


In [44]:
print("Very short duration records:", df["very_short_duration_flag"].sum())

print("\nDuration values:")
print(
    df.loc[
        df["very_short_duration_flag"],
        "trip_duration_minutes"
    ].value_counts().sort_index().head(20)
)

Very short duration records: 83746

Duration values:
trip_duration_minutes
0.000000    45069
0.016667       32
0.033333      238
0.050000     1269
0.066667     3204
0.083333     3549
0.100000     3424
0.116667     2483
0.133333     2157
0.150000     1621
0.166667     1436
0.183333     1396
0.200000     1443
0.216667     1190
0.233333      984
0.250000      864
0.266667      737
0.283333      657
0.300000      570
0.316667      521
Name: count, dtype: int64


In [45]:
zero_duration_records = df[
    df["trip_duration_minutes"] == 0
]

print("Zero-duration count:", len(zero_duration_records))

print(
    zero_duration_records[
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "fare_amount",
            "trip_distance",
            "trip_duration_minutes"
        ]
    ].head(10)
)

Zero-duration count: 45069
    tpep_pickup_datetime tpep_dropoff_datetime  fare_amount  trip_distance  \
16   2026-01-01 00:45:15   2026-01-01 00:45:15         28.9           1.54   
126  2026-01-01 00:46:05   2026-01-01 00:46:05         18.4           1.79   
251  2026-01-01 00:22:40   2026-01-01 00:22:40          8.6           1.24   
252  2026-01-01 00:36:04   2026-01-01 00:36:04         11.4           2.00   
253  2026-01-01 00:51:23   2026-01-01 00:51:23          6.5           0.83   
287  2026-01-01 00:48:27   2026-01-01 00:48:27         27.5           3.83   
382  2026-01-01 00:45:00   2026-01-01 00:45:00         27.5           5.38   
405  2026-01-01 00:51:33   2026-01-01 00:51:33         10.0           1.22   
511  2026-01-01 00:40:03   2026-01-01 00:40:03         19.8           1.69   
527  2026-01-01 00:36:07   2026-01-01 00:36:07          7.9           1.13   

     trip_duration_minutes  
16                     0.0  
126                    0.0  
251                    0.0 

In [46]:
duration_summary = pd.Series({
    "Zero duration": (df["trip_duration_minutes"] == 0).sum(),
    "Positive duration below 1 min": (
        (df["trip_duration_minutes"] > 0) &
        (df["trip_duration_minutes"] < 1)
    ).sum(),
    "Duration 1 min or more": (
        df["trip_duration_minutes"] >= 1
    ).sum()
})

print(duration_summary)
print("\nPercentage:")
print(duration_summary / len(df) * 100)

Zero duration                      45069
Positive duration below 1 min      38677
Duration 1 min or more           3641142
dtype: int64

Percentage:
Zero duration                     1.209942
Positive duration below 1 min     1.038340
Duration 1 min or more           97.751718
dtype: float64


## Trip categorization

In [47]:
distance_bins = [0, 2, 5, 10, np.inf]
distance_labels = ["Short", "Medium", "Long", "Very Long"]

df["distance_category"] = pd.cut(
    df["trip_distance"],
    bins=distance_bins,
    labels=distance_labels,
    right=False
)

In [48]:
print("Counts:")
print(df["distance_category"].value_counts().sort_index())

print("\nPercentages:")
print(
    df["distance_category"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

Counts:
distance_category
Short        2004188
Medium       1023775
Long          415690
Very Long     281235
Name: count, dtype: int64

Percentages:
distance_category
Short        53.805322
Medium       27.484719
Long         11.159799
Very Long     7.550160
Name: proportion, dtype: float64


In [49]:
print("Uncategorized records:", df["distance_category"].isna().sum())

Uncategorized records: 0


In [50]:
distance_category_summary = (
    df.groupby("distance_category", observed=True)
    .agg(
        trip_count=("trip_distance", "size"),
        avg_distance=("trip_distance", "mean"),
        median_distance=("trip_distance", "median"),
        avg_fare=("fare_amount", "mean"),
        median_fare=("fare_amount", "median"),
        avg_duration=("trip_duration_minutes", "mean"),
        median_duration=("trip_duration_minutes", "median"),
        avg_total=("total_amount", "mean"),
        median_total=("total_amount", "median")
    )
)

print(distance_category_summary)

                   trip_count  avg_distance  median_distance   avg_fare  \
distance_category                                                         
Short                 2004188      1.045748             1.06  12.449473   
Medium                1023775      3.059043             2.87  21.326668   
Long                   415690      7.125781             6.92  34.101103   
Very Long              281235     56.382699            15.30  58.787804   

                   median_fare  avg_duration  median_duration  avg_total  \
distance_category                                                          
Short                    10.24     10.131467         8.666667  19.507868   
Medium                   19.80     18.881016        17.200000  29.124462   
Long                     33.10     27.836242        24.733333  44.379703   
Very Long                59.70     45.641910        40.416667  75.823321   

                   median_total  
distance_category                
Short                   

In [51]:
very_long_distance = df.loc[
    df["distance_category"] == "Very Long",
    "trip_distance"
]

print(
    very_long_distance.quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 1.00]
    )
)

0.500        15.3000
0.750        18.0400
0.900        20.2900
0.950        22.9000
0.990        32.1366
0.995        38.9000
0.999        66.7200
1.000    269097.4800
Name: trip_distance, dtype: float64


In [52]:
df.loc[
    df["distance_category"] == "Very Long",
    [
        "trip_distance",
        "trip_duration_minutes",
        "fare_amount",
        "total_amount",
        "average_speed_mph",
        "extreme_distance_flag"
    ]
].sort_values(
    "trip_distance",
    ascending=False
).head(15)

,trip_distance,trip_duration_minutes,fare_amount,total_amount,average_speed_mph,extreme_distance_flag
3524922,269097.48,8.0,12.19,16.94,2.018231e+06,True
3632366,237567.19,38.0,72.99,81.95,3.751061e+05,True
3285749,236216.69,9.0,13.58,21.33,1.574778e+06,True
3131675,236183.06,17.0,22.75,34.96,8.335873e+05,True
2704618,221544.91,24.0,31.21,39.55,5.538623e+05,True
2688273,221436.73,11.0,27.61,32.36,1.207837e+06,True
3131405,196490.48,17.0,33.59,38.34,6.934958e+05,True
3571686,192081.68,35.0,60.20,64.95,3.292829e+05,True
2820950,184908.33,5.0,12.18,16.93,2.218900e+06,True
3499855,183363.75,84.0,71.01,75.76,1.309741e+05,True


#### Trip distance categories show an increasing pattern in median fare, total amount, and trip duration. Short trips represent approximately 53.81% of records, while Short and Medium trips together account for approximately 81.29%. The Very Long category has a substantially higher mean distance than median distance, indicating a strongly right-skewed distribution that requires further investigation.

#### The Very Long category has a median distance of 15.3 miles, but its mean is substantially inflated by extreme-distance observations. The largest observed distances are flagged as extreme anomalies. Therefore, the median is more representative of a typical trip in this category, and extreme observations should be excluded from analyses where they distort the result, while remaining preserved in the original dataset.

In [53]:
df_behavior = df[
    (df["fare_amount"] > 0) &
    (df["trip_distance"] > 0) &
    (df["trip_duration_minutes"] >= 1) &
    (~df["extreme_distance_flag"]) &
    (~df["long_duration_flag"])
].copy()

In [54]:
behavior_summary = (
    df_behavior.groupby("distance_category", observed=True)
    .agg(
        trip_count=("trip_distance", "size"),
        median_fare=("fare_amount", "median"),
        median_duration=("trip_duration_minutes", "median"),
        median_total=("total_amount", "median"),
        median_speed=("average_speed_mph", "median")
    )
)

print(behavior_summary)

                   trip_count  median_fare  median_duration  median_total  \
distance_category                                                           
Short                 1817870         10.0         8.683333         17.64   
Medium                1006564         19.8        17.283333         27.68   
Long                   270733         30.3        23.916667         38.95   

                   median_speed  
distance_category                
Short                  7.739130  
Medium                10.192771  
Long                  15.413490  


In [55]:
print(
    pd.crosstab(
        df_behavior["pickup_period"],
        df_behavior["distance_category"],
        normalize="index"
    ).mul(100).round(2)
)

distance_category  Short  Medium   Long
pickup_period                          
Afternoon          64.40   28.68   6.91
Evening            62.90   30.25   6.85
Morning            57.42   32.87   9.72
Night              48.93   39.10  11.97


## Interpretation:
- Afternoon has the largest Short-trip share: 64.40%.
- Night has the smallest Short-trip share: 48.93%.
- Night has the largest Medium and Long shares: 39.10% and 11.97%.
- Morning has a higher Long-trip share (9.72%) than Afternoon and Evening.